# Export Data: Snapshot Pipeline and File Exports

## Overview

~~Export your graph data to files and understand the snapshot export pipeline.~~

> **SNAPSHOT FUNCTIONALITY DISABLED**
> 
> This notebook tests explicit snapshot creation which has been disabled.
> Instances are now created directly from mappings using `create_from_mapping()` 
> without requiring explicit snapshots. The snapshot export happens automatically
> in the background.
> 
> **All tests in this notebook are SKIPPED.**
> 
> See `test_create_from_mapping.py` for the new instance creation workflow.

---

**Original Test Coverage (now disabled):**
- ~~Single node and multi-entity exports~~
- ~~Export job tracking and progress callbacks~~
- ~~Query failure handling and snapshot retry~~
- ~~Concurrent snapshot processing~~

## Setup

### Test Data Isolation Strategy

1. **Unique Prefixes**: Created resources use `ExportTest-{uuid}` prefix.
2. **Fresh Mapping**: Creates a new mapping for export tests.
3. **Cleanup**: Automatic cleanup via atexit.

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# All tests in this notebook are skipped because explicit snapshot creation is disabled.
# Instances are now created directly from mappings via create_from_mapping().

import pytest
pytest.skip("SNAPSHOT FUNCTIONALITY DISABLED - explicit snapshot creation is no longer supported", allow_module_level=True)

In [ ]:
# Create test context as analyst Alice
ctx = notebook.test("ExportTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

print(f"Connected to {client._config.api_url}")
print(f"Test ID: {ctx.run_id}")

In [ ]:
# Define node and edge for test mapping
person_node = NodeDefinition(
    label="Person",
    sql='SELECT id, name, age FROM bigquery.graph_olap_e2e.persons',
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql='SELECT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships',
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ]
)

# Create the main test mapping (auto-named, auto-tracked)
test_mapping = ctx.mapping(
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)
TEST_MAPPING_ID = test_mapping.id

print(f"Created test mapping: {test_mapping.name} (id={TEST_MAPPING_ID})")

## Test 3.1: Single Node Export

Create a snapshot with one node type and verify export succeeds.

In [ ]:
# Create a mapping with only one node type (no edges)
single_node_mapping = ctx.mapping(
    name=f"{ctx.prefix}-SingleNode-{ctx.run_id}",
    node_definitions=[person_node],
    edge_definitions=[],
)

print(f"Created single node mapping: {single_node_mapping.name} (id={single_node_mapping.id})")

In [ ]:
# Create snapshot from single node mapping (not using ctx.snapshot since we want non-blocking)
snapshot_3_1_name = f"{ctx.prefix}-Snapshot-3.1-{ctx.run_id}"

snapshot_3_1 = client.snapshots.create(
    mapping_id=single_node_mapping.id,
    name=snapshot_3_1_name,
    description="Single node export test",
)

# Track for cleanup
ctx._track('snapshot', snapshot_3_1.id, snapshot_3_1_name)

assert snapshot_3_1.status in ["pending", "exporting"], f"Expected pending/exporting, got {snapshot_3_1.status}"
print(f"Export 3.1 PASSED: Created snapshot {snapshot_3_1.id} with status={snapshot_3_1.status}")

In [ ]:
# Wait for snapshot to complete (with timeout)
try:
    snapshot_3_1 = client.snapshots.wait_until_ready(snapshot_3_1.id, timeout=120, poll_interval=3)
    assert snapshot_3_1.status == "ready", f"Expected status 'ready', got '{snapshot_3_1.status}'"
    print(f"Export 3.1 PASSED: Snapshot completed with status={snapshot_3_1.status}")
except Exception:
    # Check if export worker is deployed
    snapshot_check = client.snapshots.get(snapshot_3_1.id)
    if snapshot_check.status == "pending":
        print(f"Export 3.1 SKIPPED: Export worker not processing (status={snapshot_check.status})")
        print("  This test requires export worker to be deployed and running.")
    else:
        raise

## Test 3.2: Multi-Entity Export

Create a snapshot with multiple node types and edge types.

In [ ]:
# Create snapshot from multi-entity mapping (Person nodes + KNOWS edges)
snapshot_3_2_name = f"{ctx.prefix}-Snapshot-3.2-{ctx.run_id}"

snapshot_3_2 = client.snapshots.create(
    mapping_id=TEST_MAPPING_ID,
    name=snapshot_3_2_name,
    description="Multi-entity export test",
)

# Track for cleanup
ctx._track('snapshot', snapshot_3_2.id, snapshot_3_2_name)

assert snapshot_3_2.status in ["pending", "exporting"], f"Expected pending/exporting, got {snapshot_3_2.status}"
print(f"Export 3.2 PASSED: Created multi-entity snapshot {snapshot_3_2.id}")

In [ ]:
# Wait for multi-entity snapshot
try:
    snapshot_3_2 = client.snapshots.wait_until_ready(snapshot_3_2.id, timeout=120, poll_interval=3)
    assert snapshot_3_2.status == "ready"
    print("Export 3.2 PASSED: Multi-entity snapshot completed")
except Exception:
    snapshot_check = client.snapshots.get(snapshot_3_2.id)
    if snapshot_check.status == "pending":
        print("Export 3.2 SKIPPED: Export worker not processing")
    else:
        raise

## Test 3.3: Export Job Tracking

Verify export jobs are created for each entity in the snapshot.

In [ ]:
# Check progress for snapshot 3.2 which has both nodes and edges
try:
    progress = client.snapshots.get_progress(snapshot_3_2.id)
    
    # Progress should have info about export jobs
    assert progress is not None, "Progress should not be None"
    
    # Check that progress tracks node and edge exports
    print(f"Export 3.3: Progress data: {progress}")
    
    # If snapshot is ready, verify we have tracking info
    if snapshot_3_2.status == "ready":
        print("Export 3.3 PASSED: Export jobs tracked for snapshot")
    else:
        print(f"Export 3.3 PARTIAL: Progress available, status={snapshot_3_2.status}")
except NotFoundError:
    print("Export 3.3 SKIPPED: Progress endpoint not available")

## Test 3.4: Node/Edge Counts Populated

Verify that node and edge counts are populated after export completes.

In [ ]:
# Check counts in completed snapshot
if snapshot_3_2.status == "ready":
    # Refresh snapshot to get latest counts
    snapshot_3_2 = client.snapshots.get(snapshot_3_2.id)
    
    # Verify node counts exist
    assert snapshot_3_2.node_counts is not None, "Node counts should be populated"
    assert isinstance(snapshot_3_2.node_counts, dict), "Node counts should be a dict"
    assert "Person" in snapshot_3_2.node_counts, "Should have Person node count"
    assert snapshot_3_2.node_counts["Person"] > 0, "Person count should be > 0"
    
    # Verify edge counts exist
    assert snapshot_3_2.edge_counts is not None, "Edge counts should be populated"
    assert isinstance(snapshot_3_2.edge_counts, dict), "Edge counts should be a dict"
    assert "KNOWS" in snapshot_3_2.edge_counts, "Should have KNOWS edge count"
    assert snapshot_3_2.edge_counts["KNOWS"] > 0, "KNOWS count should be > 0"
    
    print(f"Export 3.4 PASSED: Node counts: {snapshot_3_2.node_counts}")
    print(f"                  Edge counts: {snapshot_3_2.edge_counts}")
else:
    print(f"Export 3.4 SKIPPED: Snapshot not ready (status={snapshot_3_2.status})")

## Test 3.5: Query Failure Fails Snapshot

Configure starburst-mock to fail queries and verify snapshot status becomes 'failed'.

In [ ]:
# Create a mapping with invalid SQL that will cause export to fail
# Node with SQL that references a non-existent table
# This will fail when the export worker tries to execute the query
invalid_node = NodeDefinition(
    label="InvalidNode",
    sql='SELECT id, name FROM analytics.nonexistent_table_that_will_fail',
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
    ]
)

fail_mapping = ctx.mapping(
    name=f"{ctx.prefix}-FailMapping-{ctx.run_id}",
    node_definitions=[invalid_node],
    edge_definitions=[],
)

print(f"Created fail mapping: {fail_mapping.name} (id={fail_mapping.id})")

In [ ]:
# Create snapshot from the fail mapping
snapshot_3_5_name = f"{ctx.prefix}-Snapshot-3.5-{ctx.run_id}"

snapshot_3_5 = client.snapshots.create(
    mapping_id=fail_mapping.id,
    name=snapshot_3_5_name,
    description="Snapshot for query failure test",
)

# Track for cleanup
ctx._track('snapshot', snapshot_3_5.id, snapshot_3_5_name)

print(f"Created snapshot {snapshot_3_5.id} with invalid SQL")

In [ ]:
# Wait for snapshot to fail (should fail when export worker tries to execute invalid SQL)
try:
    # Poll for status changes (up to 120 seconds)
    timeout = 120
    poll_interval = 3
    start_time = time.time()
    
    while time.time() - start_time < timeout:
        snapshot_3_5 = client.snapshots.get(snapshot_3_5.id)
        
        if snapshot_3_5.status == "failed":
            # Success! The snapshot failed as expected
            print("Export 3.5 PASSED: Snapshot failed as expected")
            print(f"  Status: {snapshot_3_5.status}")
            if hasattr(snapshot_3_5, 'error_message') and snapshot_3_5.error_message:
                print(f"  Error: {snapshot_3_5.error_message[:200]}")
            break
        elif snapshot_3_5.status == "ready":
            # Unexpected - snapshot succeeded despite invalid SQL
            print(f"Export 3.5 FAILED: Snapshot succeeded despite invalid SQL (status={snapshot_3_5.status})")
            break
        elif snapshot_3_5.status in ["pending", "exporting"]:
            # Still processing, keep waiting
            time.sleep(poll_interval)
        else:
            # Unknown status
            print(f"Export 3.5 WARNING: Unexpected status={snapshot_3_5.status}")
            break
    else:
        # Timeout
        snapshot_3_5 = client.snapshots.get(snapshot_3_5.id)
        if snapshot_3_5.status == "pending":
            print(f"Export 3.5 SKIPPED: Export worker not processing (status={snapshot_3_5.status})")
        else:
            print(f"Export 3.5 TIMEOUT: Snapshot still {snapshot_3_5.status} after {timeout}s")
            
except SnapshotFailedError as e:
    # The wait_until_ready might throw this exception
    print("Export 3.5 PASSED: Snapshot failed as expected (exception caught)")
    print(f"  Error: {str(e)[:200]}")
except Exception as e:
    # Check current status
    snapshot_3_5 = client.snapshots.get(snapshot_3_5.id)
    if snapshot_3_5.status == "failed":
        print(f"Export 3.5 PASSED: Snapshot failed as expected (status={snapshot_3_5.status})")
    elif snapshot_3_5.status == "pending":
        print("Export 3.5 SKIPPED: Export worker not processing")
    else:
        print(f"Export 3.5 ERROR: Unexpected error - {e}")
        raise

## Test 3.8: Snapshot Retry After Failure

Fix the mapping from test 3.5 and retry the failed snapshot.

In [ ]:
# Test 3.8: Retry failed snapshot
# Only run this test if snapshot 3.5 actually failed
if snapshot_3_5.status == "failed":
    print("Test 3.8: Retrying failed snapshot...")
    
    # Fix the mapping by updating it with valid SQL
    fixed_mapping = client.mappings.update(
        fail_mapping.id,
        change_description="Fix invalid SQL for retry test",
        node_definitions=[person_node],  # Use valid person node definition
        edge_definitions=[],
    )
    
    print(f"  Fixed mapping (now version {fixed_mapping.current_version})")
    
    # Retry the failed snapshot
    retried_snapshot = client.snapshots.retry(snapshot_3_5.id)
    
    assert retried_snapshot.id == snapshot_3_5.id, "Retry should return same snapshot ID"
    assert retried_snapshot.status in ["pending", "exporting"], \
        f"Retried snapshot should be pending/exporting, got {retried_snapshot.status}"
    
    print(f"  Retried snapshot {retried_snapshot.id} (status={retried_snapshot.status})")
    
    # Wait for retry to complete
    try:
        retried_snapshot = client.snapshots.wait_until_ready(retried_snapshot.id, timeout=120, poll_interval=3)
        
        if retried_snapshot.status == "ready":
            print("Export 3.8 PASSED: Snapshot retry succeeded")
            print(f"  Status: {retried_snapshot.status}")
            print(f"  Node counts: {retried_snapshot.node_counts}")
        elif retried_snapshot.status == "failed":
            print("Export 3.8 FAILED: Retry still failed")
            if hasattr(retried_snapshot, 'error_message'):
                print(f"  Error: {retried_snapshot.error_message[:200]}")
        else:
            print(f"Export 3.8 PARTIAL: Retry status={retried_snapshot.status}")
    except Exception as e:
        # Check status
        retried_snapshot = client.snapshots.get(retried_snapshot.id)
        if retried_snapshot.status == "pending":
            print("Export 3.8 SKIPPED: Export worker not processing retry")
        else:
            print(f"Export 3.8 ERROR: {e}")
            raise
else:
    print(f"Export 3.8 SKIPPED: Snapshot 3.5 did not fail (status={snapshot_3_5.status}), cannot test retry")

## Test 3.6: Cancel In-Progress Snapshot

Start an export and cancel it during processing.

In [ ]:
# Note: Cancel functionality may not be fully implemented
# For now, we test that deleting a pending/exporting snapshot works

snapshot_3_6_name = f"{ctx.prefix}-Snapshot-3.6-{ctx.run_id}"

snapshot_3_6 = client.snapshots.create(
    mapping_id=TEST_MAPPING_ID,
    name=snapshot_3_6_name,
    description="Snapshot for cancel test",
)

# Track for cleanup
ctx._track('snapshot', snapshot_3_6.id, snapshot_3_6_name)

print(f"Created snapshot {snapshot_3_6.id} for cancel test")

In [ ]:
# Try to delete/cancel the snapshot while it's in progress
try:
    # Give it a moment to start processing
    time.sleep(2)
    
    # Check status
    snapshot_3_6 = client.snapshots.get(snapshot_3_6.id)
    initial_status = snapshot_3_6.status
    
    # Delete (cancel) the snapshot
    client.snapshots.delete(snapshot_3_6.id)
    
    # Verify it's gone
    try:
        client.snapshots.get(snapshot_3_6.id)
        print("Export 3.6 FAILED: Snapshot should have been deleted")
    except NotFoundError:
        print(f"Export 3.6 PASSED: Snapshot cancelled/deleted from status={initial_status}")
        
except Exception as e:
    # If delete fails, it might be because snapshot completed or cancel isn't supported
    snapshot_3_6 = client.snapshots.get(snapshot_3_6.id)
    if snapshot_3_6.status == "ready":
        print(f"Export 3.6 PARTIAL: Snapshot completed before cancel ({e})")
        # Clean up
        client.snapshots.delete(snapshot_3_6.id)
    else:
        print(f"Export 3.6 SKIPPED: Cancel not supported ({e})")

## Test 3.7: Concurrent Snapshots

Create two snapshots concurrently and verify they complete independently.

In [ ]:
# Create two snapshots concurrently
snapshot_3_7a_name = f"{ctx.prefix}-Snapshot-3.7a-{ctx.run_id}"
snapshot_3_7b_name = f"{ctx.prefix}-Snapshot-3.7b-{ctx.run_id}"

snapshot_3_7a = client.snapshots.create(
    mapping_id=TEST_MAPPING_ID,
    name=snapshot_3_7a_name,
    description="Concurrent test A",
)

snapshot_3_7b = client.snapshots.create(
    mapping_id=single_node_mapping.id,
    name=snapshot_3_7b_name,
    description="Concurrent test B",
)

# Track for cleanup
ctx._track('snapshot', snapshot_3_7a.id, snapshot_3_7a_name)
ctx._track('snapshot', snapshot_3_7b.id, snapshot_3_7b_name)

print(f"Created concurrent snapshots: {snapshot_3_7a.id} and {snapshot_3_7b.id}")

In [ ]:
# Wait for both to complete (concurrently)
def wait_for_snapshot(snapshot_id, timeout=120):
    """Wait for a single snapshot."""
    return client.snapshots.wait_until_ready(snapshot_id, timeout=timeout, poll_interval=3)

try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        future_a = executor.submit(wait_for_snapshot, snapshot_3_7a.id)
        future_b = executor.submit(wait_for_snapshot, snapshot_3_7b.id)
        
        # Wait for both
        result_a = future_a.result(timeout=180)
        result_b = future_b.result(timeout=180)
        
        assert result_a.status == "ready", f"Snapshot A: expected 'ready', got '{result_a.status}'"
        assert result_b.status == "ready", f"Snapshot B: expected 'ready', got '{result_b.status}'"
        
        print("Export 3.7 PASSED: Both snapshots completed independently")
        print(f"  Snapshot A (id={result_a.id}): {result_a.status}")
        print(f"  Snapshot B (id={result_b.id}): {result_b.status}")
        
except Exception:
    # Check current status of both
    snapshot_3_7a = client.snapshots.get(snapshot_3_7a.id)
    snapshot_3_7b = client.snapshots.get(snapshot_3_7b.id)
    
    if snapshot_3_7a.status == "pending" or snapshot_3_7b.status == "pending":
        print("Export 3.7 SKIPPED: Export worker not processing")
        print(f"  Snapshot A: {snapshot_3_7a.status}")
        print(f"  Snapshot B: {snapshot_3_7b.status}")
    else:
        raise

## Test 3.9: on_progress Callback During Export

Test that the `on_progress` callback is invoked during `create_and_wait()`.

In [ ]:
# Test: on_progress callback is invoked during snapshot creation
snapshot_3_9_name = f"{ctx.prefix}-Snapshot-3.9-{ctx.run_id}"

# Track progress callbacks
export_progress_callbacks = []

def export_progress_callback(phase: str, completed: int, total: int):
    """Callback to track export progress."""
    export_progress_callbacks.append({
        'phase': phase,
        'completed': completed,
        'total': total
    })
    print(f"  Progress: {phase} ({completed}/{total})")

print(f"Creating snapshot '{snapshot_3_9_name}' with progress callback...")
try:
    snapshot_3_9 = client.snapshots.create_and_wait(
        mapping_id=TEST_MAPPING_ID,
        name=snapshot_3_9_name,
        timeout=120,
        poll_interval=3,
        on_progress=export_progress_callback,
    )
    
    # Track for cleanup
    ctx._track('snapshot', snapshot_3_9.id, snapshot_3_9_name)
    
    # Verify progress callback was called
    assert len(export_progress_callbacks) > 0, "Progress callback should have been called at least once"
    
    # Verify callback received valid data
    for cb in export_progress_callbacks:
        assert 'phase' in cb, "Callback should receive phase"
        assert 'completed' in cb, "Callback should receive completed jobs"
        assert 'total' in cb, "Callback should receive total jobs"
        assert isinstance(cb['phase'], str), "Phase should be string"
        assert isinstance(cb['completed'], int), "Completed should be int"
        assert isinstance(cb['total'], int), "Total should be int"
    
    print("Export 3.9 PASSED: on_progress callback was invoked during export")
    print(f"  Callback invocations: {len(export_progress_callbacks)}")
    print(f"  Final phase: {export_progress_callbacks[-1]['phase'] if export_progress_callbacks else 'N/A'}")
    print(f"  Final snapshot status: {snapshot_3_9.status}")

except Exception as e:
    # If export worker isn't running, snapshot may stay pending
    print(f"Export 3.9 SKIPPED: Export may not be processing - {e}")
    if len(export_progress_callbacks) > 0:
        print(f"  (Callback was invoked {len(export_progress_callbacks)} times before failure)")

## Summary

In [ ]:
# Cleanup is automatic via atexit - NotebookTest handles it
# Close client
client.close()

print("\n" + "="*60)
print("EXPORT PIPELINE E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  3.1 Single node export - Create and export single node type")
print("  3.2 Multi-entity export - Export nodes and edges together")
print("  3.3 Export job tracking - Progress tracking for exports")
print("  3.4 Node/edge counts - Verify counts after export")
print("  3.5 Query failure - Starburst error fails snapshot")
print("  3.6 Cancel snapshot - Delete/cancel in-progress export")
print("  3.7 Concurrent snapshots - Independent completion")
print("  3.8 Snapshot retry - Retry failed snapshot after fixing mapping")
print("  3.9 on_progress callback - Callback invoked during create_and_wait")
print("\nNote: Tests require export worker deployment for full coverage.")
print("\nAll test resources cleaned up automatically via atexit")